# Abstract

- Goal: Code basic Collaborative Filtering and make interactive form for it

- Dataset: [MovieLens 100K Dataset (Kaggle)](https://www.kaggle.com/datasets/prajitdatta/movielens-100k-dataset)

- Project Details:

    I coded a basic Collaborative Filtering algorithm. I use a Similarity matrix for all users for prediction and recommendation. I calculate a similarity matrix with `sklearn.metrics.pairwise.cosine_similarity`.

  
- Best result: 1.0176 (RMSE Score of Collaborative Filtering)

- Sections:
    - [Imports](#Imports)
    - [Utils](#Utils)
    - [Dataset](#Dataset)
    - [Modeling](#Modeling)
    - [Prediction](#Prediction)
        - [Setup Form](#Setup-Form)
        - [Prediction Form](#Prediction-Form)

# Imports

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import root_mean_squared_error, mean_absolute_error
from itertools import combinations
from IPython.display import display, clear_output
import ipywidgets as widgets

# Utils

In [2]:
def get_test_results(model, test_df):
    y_true = []
    y_pred = []
    for _, row in test_df.iterrows():
        user_id = row["user_id"]
        movie_id = row["item_id"]
        true_rating = row["rating"]
        try:
            pred_rating = model.predict(user_id, movie_id)
            if pred_rating != 0:
                y_true.append(true_rating)
                y_pred.append(pred_rating)
        except KeyError:
            continue
    return y_true, y_pred

In [3]:
def find_movie_by_id(id_, select_col, movie_df):
    return movie_df.iloc[id_-1][select_col]

# Dataset

In [4]:
# Load Rating Dataset
df = pd.read_csv("./data/u.data", delimiter="\t")
df.columns = ["user_id", "item_id", "rating", "timestamp"]

In [5]:
# Load Movie Info Dataset
movie_df = pd.read_csv('./data/u.item', sep="|", encoding='latin-1', header=None)
movie_df.columns = ['movie id', 'movie title' ,'release date','video release date', 'IMDb URL', 'unknown', 'Action', 
                'Adventure', 'Animation', 'Children\'s', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 
                'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']

In [6]:
df.head()

,user_id,item_id,rating,timestamp
0,186,302,3,891717742
1,22,377,1,878887116
2,244,51,2,880606923
3,166,346,1,886397596
4,298,474,4,884182806


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99999 entries, 0 to 99998
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   user_id    99999 non-null  int64
 1   item_id    99999 non-null  int64
 2   rating     99999 non-null  int64
 3   timestamp  99999 non-null  int64
dtypes: int64(4)
memory usage: 3.1 MB


In [8]:
# Formation of the Rating Matrix
rating_matrix = df.pivot(index="user_id", columns="item_id", values="rating")
rating_matrix

item_id,1,2,3,4,5,6,7,8,9,10,...,1673,1674,1675,1676,1677,1678,1679,1680,1681,1682
user_id,,,,,,,,,,,,,,,,,,,,,
1,5.0,3.0,4.0,3.0,3.0,5.0,4.0,1.0,5.0,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
939,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
940,NaN,NaN,NaN,2.0,NaN,NaN,4.0,5.0,3.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
941,5.0,NaN,NaN,NaN,NaN,NaN,4.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
# Train Test Split
train_df, test_df = df.iloc[:-20_000], df.iloc[-20_000:]
print(f"{train_df.shape=}, {test_df.shape=}")

train_df.shape=(79999, 4), test_df.shape=(20000, 4)


In [10]:
# Formation of the Train Rating Matrix
train_matrix = train_df.pivot(index="user_id", columns="item_id", values="rating")
train_matrix

item_id,1,2,3,4,5,6,7,8,9,10,...,1662,1663,1664,1666,1669,1672,1673,1677,1678,1679
user_id,,,,,,,,,,,,,,,,,,,,,
1,5.0,3.0,NaN,3.0,3.0,5.0,4.0,1.0,5.0,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
939,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
940,NaN,NaN,NaN,2.0,NaN,NaN,NaN,5.0,3.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
941,NaN,NaN,NaN,NaN,NaN,NaN,4.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Modeling

**Collaborative Filtering** is an intuitive recommendation approach based on the idea that **"similar users tend to like similar content."** To measure how similar users are, we compute **cosine similarity** between their rating vectors. The algorithm relies on a **rating matrix**, where each row represents a user, each column represents a movie, and the values are the ratings given by users to movies.

To make a prediction, we **identify users most similar to the target user** and calculate a weighted average of their ratings for a given movie, using their similarity scores as weights. This weighted sum is then **normalized** by dividing it by the sum of the similarities. To recommend movies, we predict ratings for multiple movies and select those with **the highest predicted scores**.

However, **this method has limitations**: it requires users to provide ratings. If a user only watches but **doesn't rate content**, or if the user is new (the **cold start problem**), collaborative filtering **may not work effectively**. In such cases, **content-based filtering** or **hybrid methods** (which combine both approaches) can be used to **make recommendations**.

Video from which I learned about collaborative filtering: [Collaborative Filtering : Data Science Concepts (ritvikmath)](https://youtu.be/Fmtorg_dmM0?si=wHyg-mtl-TUhxCXr)

In [11]:
class SimilarityMatrix:
    def __init__(self):
        self.similarity_matrix = None
    def calculate_matrix(self, rating_matrix):
        prepared = rating_matrix.fillna(0).to_numpy()
        self.similarity_matrix = cosine_similarity(prepared)
        return self.similarity_matrix

In [12]:
%%time
sm = SimilarityMatrix()
similarity_matrix = sm.calculate_matrix(rating_matrix)

CPU times: user 160 ms, sys: 71.9 ms, total: 232 ms
Wall time: 26.1 ms


In [13]:
class ColaborativeFiltering:
    def __init__(self, k=100):
        self.similarity_matrix = None
        self.rating_matrix = None
        self.k = k

    def fit(self, rating_matrix):
        self.rating_matrix = rating_matrix.fillna(0)
        self.similarity_matrix = SimilarityMatrix().calculate_matrix(rating_matrix)
        return self
        
    def predict(self, user_id, item_id):
        # Get similarities for specific user
        user_inx = self.rating_matrix.index.get_loc(user_id)
        similarities = self.similarity_matrix[user_inx].copy()
        similarities[user_inx] = -1
        
        # Get top k similarities for specific user
        top_users_inx = np.argsort(similarities)[-self.k:][::-1]
        top_users_ids = self.rating_matrix.index[top_users_inx]

        # Calculate weighted average
        numerators = []
        denominators = []

        for neighbor_id in top_users_ids:
            rating = self.rating_matrix.loc[neighbor_id, item_id]
            if rating != 0:
                neighbor_inx = self.rating_matrix.index.get_loc(neighbor_id)
                sim = self.similarity_matrix[user_inx, neighbor_inx]
                numerators.append(sim * rating)
                denominators.append(sim)

        # Return double checked result
        denominator_sum = sum(denominators)
        if denominator_sum != 0 and not np.isnan(denominator_sum):
            return sum(numerators) / denominator_sum
        else:
            return 0.0


            
    def recommend(self, user_id, n=5):
        user_ratings = self.rating_matrix.loc[user_id]
        not_rated_items_id = user_ratings[user_ratings == 0].index
        result_ratings = {"MovieId": [], "Predicted Rating":  []}
        for item_id in not_rated_items_id:
            result_ratings["MovieId"].append(item_id)
            result_ratings["Predicted Rating"].append(self.predict(user_id, item_id))
        result_df = pd.DataFrame(result_ratings).sort_values(by=["Predicted Rating"], ascending=False).reset_index(drop=True).iloc[:n]
        return result_df

    def add_user(self, user_ratings: dict):
        # Save user_ratings to pandas Series
        new_user_series = pd.Series(0, index=self.rating_matrix.columns)
        for item_id, rating in user_ratings.items():
            if item_id in new_user_series.index:
                new_user_series[item_id] = rating
        # Add new user ratings to rating matrix
        new_user_id = self.rating_matrix.index.max() + 1
        self.rating_matrix.loc[new_user_id] = new_user_series

        # Calculate similarities between new user and other users
        all_users = self.rating_matrix.fillna(0).to_numpy()
        new_vector = all_users[-1].reshape(1, -1)
        similarities = cosine_similarity(new_vector, all_users).flatten()

        # Add new user similarities to similarity matrix
        if self.similarity_matrix is not None:
            sim_mat = self.similarity_matrix
            sim_mat = np.vstack([sim_mat, similarities[:-1]])
            new_col = np.append(similarities[:-1], 1.0).reshape(-1, 1)
            self.similarity_matrix = np.hstack([sim_mat, new_col])
        else:
            self.similarity_matrix = cosine_similarity(all_users)
    
        return new_user_id


In [14]:
cf = ColaborativeFiltering()
cf.fit(train_matrix)

In [18]:
%%time
y_true, y_pred = get_test_results(cf, test_df)
mae = mean_absolute_error(y_true, y_pred)
rmse = root_mean_squared_error(y_true, y_pred)

print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")

MAE: 0.8072
RMSE: 1.0176
CPU times: user 8.58 s, sys: 4.79 ms, total: 8.58 s
Wall time: 8.63 s


# Prediction

## Setup Form

In [16]:
movie_ids = train_matrix.columns
movie_titles = [find_movie_by_id(movie_id, "movie title", movie_df) for movie_id in movie_ids]
title2id = dict(zip(movie_titles, movie_ids))

t1_output = widgets.Output()
t1_search = widgets.Text(placeholder='Search...')
t1_dropdown = widgets.Dropdown(options=[])

def update_dropdown(change):
    text = change['new'].lower()
    filtered = [opt for opt in movie_titles if text in opt.lower()][:100]
    t1_dropdown.options = filtered or ['No matches']
    
def t1_func(b):
    movie_title = t1_dropdown.value
    movie_id = title2id[movie_title]
    with t1_output:
        clear_output()
        print(f"Id of \"{movie_title}\" is {movie_id}")
        
t1_search.observe(update_dropdown, names='value')
t1_button = widgets.Button(description="Show Movie Id")
t1_button.on_click(t1_func)
t1 = widgets.VBox([t1_search, t1_dropdown, t1_button, t1_output])

In [17]:
t2_output = widgets.Output()

t2_user_id_field = widgets.BoundedIntText(
    value=13,
    min=0,
    max=len(train_matrix.index),
    step=1,
    description='User id from dataset:',
    disabled=False,
    style={'description_width': 'initial'}
)
t2_movie_id_field = widgets.BoundedIntText(
    value=13,
    min=0,
    max=len(train_matrix.columns),
    step=1,
    description='Movie id from dataset:',
    disabled=False,
    style={'description_width': 'initial'}
)

def t2_func(b):
    user_id = t2_user_id_field.value
    movie_id = t2_movie_id_field.value
    movie_title = find_movie_by_id(movie_id, "movie title", movie_df)
    predicted_rating = cf.predict(user_id, movie_id)
    with t2_output:
        clear_output()
        print(f"Predicted rating for \"{movie_title}\" is {predicted_rating:.2f}")
        

t2_button = widgets.Button(description="Predict Rating")
t2_button.on_click(t2_func)
t2 = widgets.VBox([t2_user_id_field, t2_movie_id_field, t2_button, t2_output])

In [18]:
t3_output = widgets.Output()

t3_user_id_field = widgets.BoundedIntText(
    value=13,
    min=0,
    max=len(train_matrix.index),
    step=1,
    description='User id from dataset:',
    disabled=False,
    style={'description_width': 'initial'}
)
t3_n_movies_field = widgets.IntSlider(
    value=5,
    min=1,
    max=100,
    step=1,
    description='How many movies to recommend:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)
def t3_func(b):
    user_id = t3_user_id_field.value
    topNmovies = t3_n_movies_field.value
    recommendations = cf.recommend(user_id, n=topNmovies)
    recommendations["Movie Title"] = recommendations["MovieId"].apply(lambda movie_id: find_movie_by_id(movie_id, "movie title", movie_df))
    recommendations.index = recommendations.index+1
    with t3_output:
        clear_output()
        display(recommendations[["Movie Title", "Predicted Rating"]])
        

t3_button = widgets.Button(description="Get Recommendations")
t3_button.on_click(t3_func)
t3 = widgets.VBox([t3_user_id_field, t3_n_movies_field, t3_button, t3_output])

In [19]:
def parse_ratings(text):
    try:
        entries = text.strip().split(',')
        ratings = {}

        for entry in entries:
            movie_id, rating = entry.strip().split(':')
            ratings[int(movie_id)] = float(rating)
        return ratings
    except Exception as e:
        print("⚠️ Error in Format:", e)
        return {}

t4_output = widgets.Output()

t4_text_input = widgets.Textarea(
    placeholder='For example: 50:5, 150:1, 64:5',
    description='Ratings:',
    layout=widgets.Layout(width='600px', height='100px'),
    style={'description_width': 'initial'}
)
t4_n_movies_field = widgets.IntSlider(
    value=5,
    min=1,
    max=100,
    step=1,
    description='How many movies to recommend:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)
def t4_func(b):
    raw_text = t4_text_input.value
    topNmovies = t4_n_movies_field.value
    ratings = parse_ratings(raw_text)
    cf.add_user(ratings)
    recommendations = cf.recommend(cf.rating_matrix.index[-1], n=topNmovies)
    recommendations["Movie Title"] = recommendations["MovieId"].apply(lambda movie_id: find_movie_by_id(movie_id, "movie title", movie_df))
    recommendations.index = recommendations.index+1
    ratings_df = pd.DataFrame({"MovieId": ratings.keys(), "Your Rating": ratings.values()}, index=np.arange(1, len(ratings)+1))
    
    ratings_df["Movie Title"] = ratings_df["MovieId"].apply(lambda movie_id: find_movie_by_id(movie_id, "movie title", movie_df))
    with t4_output:
        clear_output()
        if ratings:
            display(ratings_df[["Movie Title", "Your Rating"]])
            display(recommendations[["Movie Title", "Predicted Rating"]])
        else: 
            print("⚠️ Error in Format")
instructions = widgets.HTML("<h3>Enter for textarea your ratings for films in this format movie id:rate, movie id: rate, ...</h3>", layout=widgets.Layout(width='700px'))
t4_button = widgets.Button(description="Get Recommendations")
t4_button.on_click(t4_func)
t4 = widgets.VBox([instructions, t4_text_input, t4_n_movies_field, t4_button, t4_output])

In [20]:
form = widgets.Tab()
form.children = [t1, t2, t3, t4]
form.set_title(0, 'Search for Movie Id')
form.set_title(1, 'Predict Rate')
form.set_title(2, 'User Recommendations')
form.set_title(3, 'Your Recommendations')

## Prediction Form

In [21]:
display(form)